# Neural theorem proving — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def bce(p,y):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    y=np.asarray(y,dtype=float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
print('setup ok')

## Soft unification turns proof search into graded similarity
Neural theorem proving keeps the shape of logical proof search but replaces exact symbol matching with differentiable similarity. These five examples show soft unification scores, proof-path scoring, a temperature knob, competing proof paths, and a satisfaction curve for a learned rule weight.

In [ ]:
# 1) soft unification scores symbols by embedding cosine similarity
names=['alice','bob','carol']; E=np.array([[1.,0.],[0.8,0.2],[0.,1.]]); E=E/np.linalg.norm(E,axis=1,keepdims=True); S=E@E.T
plt.figure(figsize=(4,4)); plt.imshow(S,vmin=0,vmax=1,cmap='Blues'); plt.colorbar(label='cosine'); plt.xticks(range(3),names); plt.yticks(range(3),names); plt.title('soft unification matrix')
assert abs(float(S[0,1])-0.9701425001453318)<1e-12

In [ ]:
# 2) a proof path score multiplies fact confidence and unification confidence
fact_conf=0.9; unify=float(S[0,1]); rule_conf=0.8; proof_score=fact_conf*unify*rule_conf
plt.figure(figsize=(6,3)); plt.bar(['fact','unify','rule','product'],[fact_conf,unify,rule_conf,proof_score]); plt.ylim(0,1); plt.title(f'proof score={proof_score:.4f}')
assert abs(proof_score-0.6985026001046389)<1e-12

In [ ]:
# 3) temperature sharpens or softens which facts can unify
sims=np.array([S[0,0],S[0,1],S[0,2]]); temps=np.array([0.2,0.5,1.0]); weights=[]
for tau in temps:
    z=np.exp(sims/tau); weights.append(z/z.sum())
weights=np.array(weights)
plt.figure(figsize=(6,3)); [plt.plot(names,weights[i],'-o',label=f'tau={temps[i]}') for i in range(len(temps))]; plt.legend(); plt.ylabel('softmax proof weight'); plt.title('lower temperature concentrates proof mass')
assert weights[0,0]>weights[-1,0] and weights[-1,2]>weights[0,2]

In [ ]:
# 4) competing proof paths combine by soft OR: 1-prod(1-score)
path_scores=np.array([0.6985026001046389,0.25,0.10]); combined=1-np.prod(1-path_scores)
plt.figure(figsize=(6,3)); plt.bar(['path1','path2','path3','soft OR'],list(path_scores)+[combined]); plt.ylim(0,1); plt.title(f'combined theorem score={combined:.4f}')
assert abs(float(combined)-0.7964892550706313)<1e-12

In [ ]:
# 5) increasing a rule weight raises theorem satisfaction smoothly
ws=np.linspace(0,1,51); theorem=1-np.prod(1-(ws[:,None]*np.array([unify*0.9,0.25,0.10])),axis=1)
plt.figure(figsize=(6,3)); plt.plot(ws,theorem); plt.xlabel('learned rule weight'); plt.ylabel('theorem score'); plt.title('satisfaction curve for a neural proof')
assert theorem[-1]>theorem[25]>theorem[0]